In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# 1. Setup folders and data
os.makedirs('models', exist_ok=True)
df = pd.read_csv('final_processed_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

best_models_summary = []

# 2. Define the Tournament Loop
for state in df['State'].unique():
    print(f"\n--- 🏆 Tournament for {state} ---")
    state_df = df[df['State'] == state].sort_values('Date').copy()

    # Split: last 8 weeks for testing
    train = state_df.iloc[:-8]
    test = state_df.iloc[-8:]

    # --- MODEL 1: ARIMA ---
    try:
        arima_model = SARIMAX(train['Total'], order=(1,1,1)).fit(disp=False)
        arima_pred = arima_model.forecast(steps=8)
        arima_rmse = np.sqrt(mean_squared_error(test['Total'], arima_pred))
    except: arima_rmse = float('inf')

    # --- MODEL 2: PROPHET ---
    try:
        p_df = train[['Date', 'Total']].rename(columns={'Date': 'ds', 'Total': 'y'})
        m = Prophet(yearly_seasonality=True, interval_width=0.95).fit(p_df)
        future = m.make_future_dataframe(periods=8, freq='W')
        prophet_pred = m.predict(future)['yhat'].iloc[-8:]
        prophet_rmse = np.sqrt(mean_squared_error(test['Total'], prophet_pred))
    except: prophet_rmse = float('inf')

    # --- MODEL 3: XGBOOST ---
    try:
        feats = ['lag_1', 'lag_7', 'lag_30', 'rolling_mean_4', 'month', 'week_of_year']
        xgb = XGBRegressor(n_estimators=100).fit(train[feats], train['Total'])
        xgb_pred = xgb.predict(test[feats])
        xgb_rmse = np.sqrt(mean_squared_error(test['Total'], xgb_pred))
    except: xgb_rmse = float('inf')

    # --- MODEL 4: LSTM ---
    try:
        scaler = MinMaxScaler()
        train_sc = scaler.fit_transform(train[['Total']])
        # Prepare 1-week lookback
        X_lstm = train_sc[:-1].reshape(-1, 1, 1)
        y_lstm = train_sc[1:]
        lstm = Sequential([LSTM(20, input_shape=(1, 1)), Dense(1)])
        lstm.compile(optimizer='adam', loss='mse')
        lstm.fit(X_lstm, y_lstm, epochs=10, verbose=0)
        # Simplified prediction for tournament speed
        lstm_rmse = 999999999 # Placeholder (LSTM usually needs more tuning per state)
    except: lstm_rmse = float('inf')

    # --- WINNER SELECTION ---
    results = {'ARIMA': arima_rmse, 'Prophet': prophet_rmse, 'XGBoost': xgb_rmse}
    winner = min(results, key=results.get)

    print(f"WINNER: {winner} (RMSE: {results[winner]:.2f})")
    best_models_summary.append({'State': state, 'Best_Model': winner, 'RMSE': results[winner]})

    # Save the winner
    if winner == 'ARIMA': joblib.dump(arima_model, f'models/{state}_model.pkl')
    elif winner == 'Prophet': joblib.dump(m, f'models/{state}_model.pkl')
    elif winner == 'XGBoost': joblib.dump(xgb, f'models/{state}_model.pkl')

# 3. Final Export
summary_df = pd.DataFrame(best_models_summary)
summary_df.to_csv('models/best_models_summary.csv', index=False)


--- 🏆 Tournament for Alabama ---


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


WINNER: XGBoost (RMSE: 96176605.33)

--- 🏆 Tournament for Arizona ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros

WINNER: XGBoost (RMSE: 100058318.42)

--- 🏆 Tournament for Arkansas ---


...

WINNER: XGBoost (RMSE: 25500546.98)

--- 🏆 Tournament for Wisconsin ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros

WINNER: XGBoost (RMSE: 55161826.85)

--- 🏆 Tournament for Wyoming ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


WINNER: XGBoost (RMSE: 10058367.34)


In [2]:
!zip -r models_export.zip models/

  adding: models/ (stored 0%)
  adding: models/Rhode Island_model.pkl (deflated 75%)
  adding: models/California_model.pkl (deflated 58%)
  adding: models/New Hampshire_model.pkl (deflated 75%)
  adding: models/Georgia_model.pkl (deflated 75%)
  adding: models/South Carolina_model.pkl (deflated 59%)
  adding: models/Kentucky_model.pkl (deflated 75%)
  adding: models/Washington_model.pkl (deflated 74%)
  adding: models/Oklahoma_model.pkl (deflated 59%)
  adding: models/Minnesota_model.pkl (deflated 75%)
  adding: models/Kansas_model.pkl (deflated 75%)
  adding: models/Alabama_model.pkl (deflated 75%)
  adding: models/Nevada_model.pkl (deflated 74%)
  adding: models/best_models_summary.csv (deflated 46%)
  adding: models/Florida_model.pkl (deflated 57%)
  adding: models/Indiana_model.pkl (deflated 75%)
  adding: models/Arkansas_model.pkl (deflated 75%)
  adding: models/Michigan_model.pkl (deflated 74%)
  adding: models/Wisconsin_model.pkl (deflated 74%)
  adding: models/North Carolina_mo